# Machine-learning workflow

This notebook reproduces the final machine-learning workflow reported in:

**A Causal Hybrid Framework for Microscopic BEV Energy Consumption Prediction under Real Driving Conditions**

The experimental dataset is not included. Place an authorized local copy at:

`../data/BEV_model_ready_dataset.xlsx`

This public notebook intentionally contains no saved experimental outputs.

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

import xgboost as xgb

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_PATH = Path("../data/BEV_model_ready_dataset.xlsx")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Load and normalize the model-ready dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "The experimental dataset is not publicly distributed. "
        "Place an authorized copy in the local data/ directory."
    )

df = pd.read_excel(DATA_PATH)

# Compatibility with the original working dataset names
rename_map = {
    "Speed": "v",
    "Acceleration": "a",
    "Road_gradient": "theta",
    "RPM": "MotorSpeed",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

print(f"Dataset dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique trips: {df['Trip'].nunique()}")

## 2. Final predictor set and target

In [ ]:
selected_features = [
    "Ew_grade",
    "Ew_drag",
    "Ew_roll",
    "Ew_inertia",
    "SoC",
    "BattTemp_max",
    "MotorTorque",
    "MotorTemp",
]

target = "E_batt_net"
trip_column = "Trip"

required = selected_features + [target, trip_column]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

X = df[selected_features].copy()
y = df[target].copy()
trips = df[trip_column].copy()

## 3. Trip-wise train/validation/test split

The original analysis generated this split using `random_state=42`. The public notebook hard-codes the resulting trip IDs so that reproduction does not depend on the order in which trip labels appear in a local copy of the dataset.

In [ ]:
TRAIN_TRIPS = [2, 3, 4, 5, 7, 8, 11, 13, 15]
VAL_TRIPS = [1, 6, 14]
TEST_TRIPS = [9, 10, 12]

assert set(TRAIN_TRIPS).isdisjoint(VAL_TRIPS)
assert set(TRAIN_TRIPS).isdisjoint(TEST_TRIPS)
assert set(VAL_TRIPS).isdisjoint(TEST_TRIPS)

train_mask = trips.isin(TRAIN_TRIPS)
val_mask = trips.isin(VAL_TRIPS)
test_mask = trips.isin(TEST_TRIPS)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print(f"Training observations:   {len(X_train):,}")
print(f"Validation observations: {len(X_val):,}")
print(f"Test observations:       {len(X_test):,}")

assert len(X_train) + len(X_val) + len(X_test) == len(df)

## 4. Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

## 5. SVR epsilon sensitivity analysis

`C=10` and `gamma='scale'` are fixed. Epsilon is selected only from validation-set results by minimum RMSE, with MAE used as a tie-breaker. The independent test set is not used for epsilon selection.

In [ ]:
C_FIXED = 10
GAMMA_FIXED = "scale"
EPSILON_VALUES = [0.0005, 0.001, 0.002]

svr_rows = []

for epsilon in EPSILON_VALUES:
    candidate = SVR(
        kernel="rbf",
        C=C_FIXED,
        epsilon=epsilon,
        gamma=GAMMA_FIXED,
        cache_size=4000,
    )
    candidate.fit(X_train_scaled, y_train)
    val_pred = candidate.predict(X_val_scaled)

    svr_rows.append({
        "epsilon": epsilon,
        "Val_RMSE": np.sqrt(mean_squared_error(y_val, val_pred)),
        "Val_MAE": mean_absolute_error(y_val, val_pred),
        "Val_R2": r2_score(y_val, val_pred),
    })

svr_validation = (
    pd.DataFrame(svr_rows)
    .sort_values(["Val_RMSE", "Val_MAE"], ascending=[True, True])
    .reset_index(drop=True)
)

BEST_EPSILON = float(svr_validation.loc[0, "epsilon"])

print(svr_validation)
print(f"Selected epsilon: {BEST_EPSILON}")

## 6. Model definitions

In [ ]:
models = {
    "MLR": LinearRegression(),

    "SVR": SVR(
        kernel="rbf",
        C=C_FIXED,
        epsilon=BEST_EPSILON,
        gamma=GAMMA_FIXED,
        cache_size=4000,
        verbose=False,
    ),

    "MLP": MLPRegressor(
        hidden_layer_sizes=(100, 50, 30),
        activation="relu",
        solver="adam",
        max_iter=1000,
        batch_size=256,
        random_state=RANDOM_STATE,
        verbose=False,
        early_stopping=True,
        validation_fraction=0.1,
        learning_rate="adaptive",
    ),

    "XGBoost": xgb.XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    ),
}

## 7. Training and evaluation

Unscaled predictors are used for MLR, XGBoost, and Random Forest. Standardized predictors are used for SVR and MLP, matching the final analysis.

In [ ]:
results = {}
predictions = {}
training_times = {}
feature_importances = {}

for model_name, model in models.items():
    start = time.time()

    if model_name == "MLR":
        model.fit(X_train, y_train)
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

    elif model_name in {"XGBoost", "Random Forest"}:
        if model_name == "XGBoost":
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        else:
            model.fit(X_train, y_train)

        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

        feature_importances[model_name] = {
            "importance": model.feature_importances_,
            "type": "model_native",
        }

    else:
        model.fit(X_train_scaled, y_train)
        val_pred = model.predict(X_val_scaled)
        test_pred = model.predict(X_test_scaled)

    training_times[model_name] = time.time() - start

    results[model_name] = {
        "Val_RMSE": np.sqrt(mean_squared_error(y_val, val_pred)),
        "Val_MAE": mean_absolute_error(y_val, val_pred),
        "Val_R2": r2_score(y_val, val_pred),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
        "Test_MAE": mean_absolute_error(y_test, test_pred),
        "Test_R2": r2_score(y_test, test_pred),
    }

    predictions[model_name] = {
        "val": val_pred,
        "test": test_pred,
    }

results_df = pd.DataFrame(results).T
results_df

## 8. Model selection

The manuscript selects the final model **exclusively from validation-set performance**. The independent test set is not used to retrospectively reselect a model.

In [ ]:
selected_model = (
    results_df
    .sort_values(["Val_RMSE", "Val_MAE", "Val_R2"], ascending=[True, True, False])
    .index[0]
)

print(f"Selected model based on validation performance: {selected_model}")
print(results_df.loc[selected_model])

## 9. Independent test performance by trip

This section reports generalization across the three unseen test trips without changing the previously selected model.

In [ ]:
trip_results = {}

test_trip_labels = trips.loc[test_mask].reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

for trip in TEST_TRIPS:
    local_mask = (test_trip_labels == trip).to_numpy()
    y_true_trip = y_test_reset.loc[local_mask]

    trip_results[trip] = {"N_samples": int(local_mask.sum())}

    for model_name in models:
        y_pred_trip = predictions[model_name]["test"][local_mask]

        trip_results[trip][f"{model_name}_RMSE"] = np.sqrt(
            mean_squared_error(y_true_trip, y_pred_trip)
        )
        trip_results[trip][f"{model_name}_MAE"] = mean_absolute_error(
            y_true_trip, y_pred_trip
        )
        trip_results[trip][f"{model_name}_R2"] = r2_score(
            y_true_trip, y_pred_trip
        )

trip_results_df = pd.DataFrame(trip_results).T
trip_results_df

## 10. Predictor importance

For the MLP, permutation importance is calculated on the independent test set as a **post-hoc interpretability analysis**, matching the final manuscript workflow. XGBoost and Random Forest use their model-native feature importances.

This post-hoc use of the test set does not alter model selection or model fitting.

In [ ]:
mlp_perm = permutation_importance(
    models["MLP"],
    X_test_scaled,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="r2",
    n_jobs=-1,
)

feature_importances["MLP"] = {
    "importance": np.maximum(mlp_perm.importances_mean, 0),
    "type": "permutation_importance",
}

models_for_importance = ["MLP", "XGBoost", "Random Forest"]
importance_table = pd.DataFrame({"Variable": selected_features})

for model_name in models_for_importance:
    values = np.abs(feature_importances[model_name]["importance"])
    total = values.sum()
    importance_table[model_name] = values if total == 0 else values / total

importance_table["Average"] = importance_table[models_for_importance].mean(axis=1)
importance_table = importance_table.sort_values("Average", ascending=False)
importance_table

## 11. Export aggregate summaries

In [ ]:
results_df.to_csv(OUTPUT_DIR / "model_performance.csv")
svr_validation.to_csv(OUTPUT_DIR / "svr_epsilon_sensitivity.csv", index=False)
trip_results_df.to_csv(OUTPUT_DIR / "test_performance_by_trip.csv")
importance_table.to_csv(OUTPUT_DIR / "predictor_importance.csv", index=False)

print(f"Aggregate outputs written to: {OUTPUT_DIR.resolve()}")